# Preprocessing Pipeline – Sri Lanka Used-Car Price Dataset

**Source:** Riyasewana.com (scraped via `scraper.py`)  
**Target variable:** `price_lkr` – car listing price in LKR  

### Steps
1. Load raw CSV  
2. Drop irrelevant / non-feature columns  
3. Fix data-entry errors in categorical columns  
4. Filter outliers / invalid values  
5. Handle missing values  
6. Feature engineering (`car_age`, `mileage_per_year`)  
7. Label encoding for ordinal features  
8. One-hot encoding for nominal features  
9. Min-Max normalisation for numeric features  
10. Save preprocessed CSV  

## Step 1 – Import Libraries

In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

RAW_FILE     = '../data/raw/riyasewana_cars.csv'
OUT_FILE     = '../data/processed/riyasewana_cars_preprocessed.csv'
CURRENT_YEAR = 2025

## Step 2 – Load Raw Dataset

In [40]:
df = pd.read_csv(RAW_FILE)
print(f'Raw shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Raw shape: (5129, 15)
Columns: ['title', 'make', 'model', 'year', 'price_lkr', 'mileage_km', 'location', 'fuel_type', 'transmission', 'engine_cc', 'options', 'details', 'condition', 'date_posted', 'listing_url']


,title,make,model,year,price_lkr,mileage_km,location,fuel_type,transmission,engine_cc,options,details,condition,date_posted,listing_url
0,Nissan FB15 2002 Car,Nissan,FB15,2002,3950000.0,313000.0,Kelaniya,Petrol,Automatic,1490.0,"AIR CONDITION, POWER WINDOW",Nissan FB15 – 1.5L Petrol | 2002 | Japan Key D...,NaN,2026-02-17,https://riyasewana.com/buy/nissan-fb15-sale-ke...
1,Toyota Aqua 2014 Car,Toyota,Aqua,2014,7600000.0,93000.0,Horana,Hybrid,Automatic,1500.0,"AIR CONDITION, POWER STEERING, POWER MIRROR, P...",Toyota Aqua Hybrid 2014 S Grade Price negotiat...,NaN,2026-02-11,https://riyasewana.com/buy/toyota-aqua-sale-ho...
2,Daihatsu Mira X SafetyS AA 2024 Car,Daihatsu,Mira X SafetyS AA,2024,6795000.0,10.0,Ragama,Petrol,Automatic,660.0,"AIR CONDITION, POWER STEERING, POWER WINDOW","- UNREGISTERED - / LKR 6,795,000 / 2024 YOM / ...",NaN,2026-02-13,https://riyasewana.com/buy/daihatsu-mira-x-sal...


### Missing values in raw data

In [41]:
df.isnull().sum()

title              0
make               0
model              0
year               0
price_lkr       1045
mileage_km       412
location           0
fuel_type          0
transmission       2
engine_cc        177
options            0
details            0
condition       5129
date_posted        0
listing_url        0
dtype: int64

## Step 3 – Drop Irrelevant / Unusable Columns

| Column | Reason for dropping |
|---|---|
| `title` | Free-text, redundant with make/model/year |
| `listing_url` | Unique identifier, not a feature |
| `options` | Unstructured free-text blob |
| `details` | Unstructured free-text blob |
| `date_posted` | Listing date, not a car attribute |
| `condition` | 100% missing |


In [42]:
cols_to_drop = ['title', 'listing_url', 'options', 'details', 'date_posted', 'condition']
df.drop(columns=cols_to_drop, inplace=True)
print(f'Remaining columns: {df.columns.tolist()}')

Remaining columns: ['make', 'model', 'year', 'price_lkr', 'mileage_km', 'location', 'fuel_type', 'transmission', 'engine_cc']


## Step 4 – Fix Categorical Data-Entry Errors

- `transmission` has scraping artefacts `'100'` and `'2000'` → replaced with `NaN`  
- `fuel_type` has invalid value `'Kick'` → replaced with `NaN`  
- All string columns standardised with `.str.title()`  

In [43]:
# Fix transmission artefacts
invalid_trans = df['transmission'].isin(['100', '2000'])
print(f'transmission – invalid entries replaced with NaN: {invalid_trans.sum()}')
df.loc[invalid_trans, 'transmission'] = np.nan

# Fix fuel_type
invalid_fuel = df['fuel_type'] == 'Kick'
print(f"fuel_type – 'Kick' entries replaced with NaN: {invalid_fuel.sum()}")
df.loc[invalid_fuel, 'fuel_type'] = np.nan

# Standardise capitalisation
for col in ['fuel_type', 'transmission', 'make', 'model', 'location']:
    df[col] = df[col].str.strip().str.title()

transmission – invalid entries replaced with NaN: 2
fuel_type – 'Kick' entries replaced with NaN: 1


## Step 5 – Filter Outliers & Invalid Numeric Values

- **Year:** Remove records with `year < 1970` or `year > 2025`  
- **Mileage:** Apply 3×IQR upper fence to remove extreme scraping errors (e.g. 2.5 billion km)  
- **Price:** Remove prices below 50,000 LKR (likely data errors)  

In [44]:
original_len = len(df)

# Year filter
invalid_year = (df['year'] < 1970) | (df['year'] > CURRENT_YEAR)
print(f'Removing {invalid_year.sum()} rows with year outside [1970, {CURRENT_YEAR}]')
df = df[~invalid_year]

# Mileage IQR filter
Q1 = df['mileage_km'].quantile(0.25)
Q3 = df['mileage_km'].quantile(0.75)
IQR = Q3 - Q1
mileage_upper = Q3 + 3 * IQR
invalid_mileage = df['mileage_km'] > mileage_upper
print(f'Removing {invalid_mileage.sum()} rows where mileage_km > {mileage_upper:,.0f} km (3xIQR fence)')
df = df[~invalid_mileage]

# Low price filter
low_price = df['price_lkr'] < 50_000
print(f'Removing {low_price.sum()} rows with price_lkr < 50,000')
df = df[~low_price | df['price_lkr'].isna()]

print(f'Rows after outlier removal: {len(df)}  (removed {original_len - len(df)})')

Removing 36 rows with year outside [1970, 2025]
Removing 47 rows where mileage_km > 444,000 km (3xIQR fence)
Removing 0 rows with price_lkr < 50,000
Rows after outlier removal: 5046  (removed 83)


## Step 6 – Handle Missing Values

| Column | Strategy |
|---|---|
| `price_lkr` (target) | Drop rows — cannot impute the target |
| `mileage_km` | Grouped median by `make` + `fuel_type` |
| `engine_cc` | Grouped median by `make` + `model` |
| `transmission` | Grouped mode by `make` + `model` |
| `fuel_type` | Grouped mode by `make` |


In [45]:
# Drop rows where target is missing
before = len(df)
df = df.dropna(subset=['price_lkr'])
print(f'Dropped {before - len(df)} rows where price_lkr is NaN')

# Impute mileage_km
df['mileage_km'] = df.groupby(['make', 'fuel_type'])['mileage_km'].transform(lambda x: x.fillna(x.median()))
df['mileage_km'].fillna(df['mileage_km'].median(), inplace=True)

# Impute engine_cc
df['engine_cc'] = df.groupby(['make', 'model'])['engine_cc'].transform(lambda x: x.fillna(x.median()))
df['engine_cc'].fillna(df['engine_cc'].median(), inplace=True)

# Helper: fill with mode
def fill_mode(s):
    m = s.mode()
    return s.fillna(m[0] if not m.empty else np.nan)

# Impute transmission
df['transmission'] = df.groupby(['make', 'model'])['transmission'].transform(fill_mode)
df['transmission'].fillna(df['transmission'].mode()[0], inplace=True)

# Impute fuel_type
df['fuel_type'] = df.groupby('make')['fuel_type'].transform(fill_mode)
df['fuel_type'].fillna(df['fuel_type'].mode()[0], inplace=True)

print('Missing values after imputation:')
df.isnull().sum()

Dropped 1021 rows where price_lkr is NaN


c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\USER\A

Missing values after imputation:


make            0
model           0
year            0
price_lkr       0
mileage_km      0
location        0
fuel_type       0
transmission    0
engine_cc       0
dtype: int64

## Step 7 – Feature Engineering

- **`car_age`** = `2025 - year` — how old the car is  
- **`mileage_per_year`** = `mileage_km / car_age` — driving intensity proxy  
- Original `year` column is dropped (now encoded via `car_age`)  

In [46]:
df['car_age'] = CURRENT_YEAR - df['year']
df['mileage_per_year'] = df['mileage_km'] / df['car_age'].replace(0, 1)
df.drop(columns=['year'], inplace=True)
print(f"car_age range: {df['car_age'].min()} - {df['car_age'].max()} years")
df[['car_age', 'mileage_per_year']].describe()

car_age range: 0 - 55 years


,car_age,mileage_per_year
count,4025.000000,4025.000000
mean,16.260124,8923.510230
std,10.368372,9042.323456
min,0.000000,0.000000
25%,9.000000,5472.222222
50%,14.000000,8125.000000
75%,22.000000,11142.857143
max,55.000000,165000.000000


## Step 8 – Encoding Categorical Variables

- **Label Encoding** (ordinal): `transmission` → Automatic=0, Manual=1  
- **One-Hot Encoding** (nominal): `fuel_type`, `make` (top 15), `location` (top 20)  
  - Rare makes/locations grouped as `'Other'` to avoid feature explosion  

In [47]:
# Label encode transmission
le = LabelEncoder()
df['transmission_enc'] = le.fit_transform(df['transmission'])
print(f"transmission mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Group rare makes and locations
TOP_MAKES = 15
TOP_LOCATIONS = 20

make_counts = df['make'].value_counts()
df['make_grouped'] = df['make'].where(df['make'].isin(make_counts.nlargest(TOP_MAKES).index), other='Other')

loc_counts = df['location'].value_counts()
df['location_grouped'] = df['location'].where(df['location'].isin(loc_counts.nlargest(TOP_LOCATIONS).index), other='Other')

# One-hot encode
ohe_cols = ['fuel_type', 'make_grouped', 'location_grouped']
df = pd.get_dummies(df, columns=ohe_cols, drop_first=False)

# Drop original high-cardinality columns
df.drop(columns=['make', 'model', 'location', 'transmission'], inplace=True, errors='ignore')
print(f'Shape after encoding: {df.shape}')

transmission mapping: {'Automatic': 0, 'Manual': 1}
Shape after encoding: (4025, 47)


## Step 9 – Min-Max Normalisation

Numeric features scaled to **[0, 1]** range.  
> **Note:** Target variable `price_lkr` is intentionally **NOT** scaled.  

In [48]:
numeric_cols = ['mileage_km', 'engine_cc', 'car_age', 'mileage_per_year']
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
print(f'Scaled (0-1): {numeric_cols}')

Scaled (0-1): ['mileage_km', 'engine_cc', 'car_age', 'mileage_per_year']


## Step 10 – Final Summary & Save

In [49]:
print(f'Final shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Final shape: (4025, 47)
Columns: ['price_lkr', 'mileage_km', 'engine_cc', 'car_age', 'mileage_per_year', 'transmission_enc', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_Hybrid', 'fuel_type_Petrol', 'make_grouped_Audi', 'make_grouped_Bmw', 'make_grouped_Daihatsu', 'make_grouped_Honda', 'make_grouped_Hyundai', 'make_grouped_Kia', 'make_grouped_Mazda', 'make_grouped_Mercedes-Benz', 'make_grouped_Micro', 'make_grouped_Mitsubishi', 'make_grouped_Nissan', 'make_grouped_Other', 'make_grouped_Perodua', 'make_grouped_Suzuki', 'make_grouped_Tata', 'make_grouped_Toyota', 'location_grouped_Anuradapura', 'location_grouped_Colombo', 'location_grouped_Dehiwala-Mount-Lavinia', 'location_grouped_Galle', 'location_grouped_Gampaha', 'location_grouped_Homagama', 'location_grouped_Horana', 'location_grouped_Kadawatha', 'location_grouped_Kaduwela', 'location_grouped_Kalutara', 'location_grouped_Kandy', 'location_grouped_Kelaniya', 'location_grouped_Kurunegala', 'location_grouped_Malabe', 'location_

,price_lkr,mileage_km,engine_cc,car_age,mileage_per_year,transmission_enc,fuel_type_Diesel,fuel_type_Electric,fuel_type_Hybrid,fuel_type_Petrol,...,location_grouped_Kelaniya,location_grouped_Kurunegala,location_grouped_Malabe,location_grouped_Matara,location_grouped_Moratuwa,location_grouped_Negombo,location_grouped_Other,location_grouped_Panadura,location_grouped_Piliyandala,location_grouped_Ratnapura
0,3950000.0,0.734997,2.088133e-06,0.418182,0.082477,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1,7600000.0,0.218386,2.102157e-06,0.200000,0.051240,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,6795000.0,0.000023,9.241638e-07,0.018182,0.000061,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0


In [50]:
df.describe()

,price_lkr,mileage_km,engine_cc,car_age,mileage_per_year,transmission_enc,fuel_type_Diesel,fuel_type_Electric,fuel_type_Hybrid,fuel_type_Petrol,...,location_grouped_Kelaniya,location_grouped_Kurunegala,location_grouped_Malabe,location_grouped_Matara,location_grouped_Moratuwa,location_grouped_Negombo,location_grouped_Other,location_grouped_Panadura,location_grouped_Piliyandala,location_grouped_Ratnapura
count,4.025000e+03,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,...,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000,4025.000000
mean,6.184905e+06,0.277286,0.000250,0.295639,0.054082,0.303851,0.048199,0.008944,0.150311,0.792547,...,0.016149,0.054161,0.039006,0.035776,0.014410,0.015652,0.355528,0.036522,0.021118,0.014658
std,5.869060e+06,0.160327,0.015762,0.188516,0.054802,0.459976,0.214213,0.094161,0.357420,0.405533,...,0.126064,0.226364,0.193634,0.185755,0.119188,0.124141,0.478732,0.187608,0.143796,0.120196
min,1.500000e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.200000e+06,0.178466,0.000001,0.163636,0.033165,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5.400000e+06,0.289904,0.000002,0.254545,0.049242,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,7.800000e+06,0.386841,0.000002,0.400000,0.067532,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
max,1.320000e+08,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [51]:
import os
os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f'[DONE] Preprocessed dataset saved -> {OUT_FILE}')

[DONE] Preprocessed dataset saved -> ../data/processed/riyasewana_cars_preprocessed.csv
